# Data collection: web scraping

Isse Abdallah Mohamoud. IBM Applied Data Science Capstone historical snapshots. Run from the spacex-capstone folder.

The original historical Wikipedia revision returned HTTP 403. The current launch-list page was successfully fetched (HTTP 200) on 29 October 2026. This notebook executes the parser on that retrieved HTML. The current 2025-2026 records demonstrate the workflow and are never merged into the 2010-2020 model cohort.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0,str(Path.cwd()))
import json,pandas as pd
from IPython.display import display
from evidence.scrape_workflow import main
# main requests the URL when its HTML cache is absent; status checks precede parsing.
scraped=main(refresh=False)
status=json.loads(Path('evidence/scraping_evidence.json').read_text())
display(pd.DataFrame([status])[['http_status','html_tables','completed_launch_tables','parsed_rows']])
print('Page title:',status['title'])

{
  "url": "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches",
  "http_status": 200,
  "retrieval_date": "2026-10-29",
  "title": "List of Falcon 9 and Falcon Heavy launches - Wikipedia",
  "html_tables": 19,
  "completed_launch_tables": 2,
  "parsed_rows": 285,
  "columns": [
    "Flight No.",
    "Date and time (UTC)",
    "Version, booster",
    "Launch site",
    "Payload",
    "Payload mass",
    "Orbit",
    "Customer",
    "Launch outcome",
    "Booster landing"
  ],
  "historical_revision_http_status": 403,
  "scope_note": "Current page completed launch tables (2025-2026), independent workflow demonstration. Not substituted into the 2010-2020 modeling cohort."
}
Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


,http_status,html_tables,completed_launch_tables,parsed_rows
0,200,19,2,285


In [2]:
display(scraped[['Flight No.','Date and time (UTC)','Launch site','Launch outcome','Booster landing']].head())
print('Records:',len(scraped),'Columns:',len(scraped.columns))
assert scraped['Flight No.'].is_unique
assert len(scraped)==285
print('Description rows removed; flight numbers are unique; current-page CSV saved.')

Records: 285 Columns: 10
Description rows removed; flight numbers are unique; current-page CSV saved.


,Flight No.,Date and time (UTC),Launch site,Launch outcome,Booster landing
0,418,"January 4, 2025 01:27","Cape Canaveral, SLC‑40",Success,Success (ASOG)
1,419,"January 6, 2025 20:43","Cape Canaveral, SLC‑40",Success,Success (JRTI)
2,420,"January 8, 2025 15:27","Kennedy, LC‑39A",Success,Success (ASOG)
3,421,"January 10, 2025 03:53","Vandenberg, SLC‑4E",Success,Success (OCISLY)
4,422,"January 10, 2025 19:11","Cape Canaveral, SLC‑40",Success,Success (JRTI)


scrape_workflow.py uses BeautifulSoup to locate completed-launch tables; pandas.read_html handles table structure. It filters genuine date rows, removes citations and unnamed columns, and deduplicates flight number. The cleaned CSV and retrieval metadata are committed. The large HTML cache is not required in GitHub; rerunning the request can return newer data.